In [1]:
import pandas as pd

caminho = "../data/raw/vacinabr-dados-por-municipio.csv"

usecols = [
    "ibge6", "uf", "ano", "imunizante", "faixa_etaria", "doenca",
    "populacao", "qt_dose", "cobertura", "meta", "meta_atingida",
    "abandono", "homog_vacinas", "reforco", "ultima_dose",
]

vacinas_mvp = [
    "BCG",
    "Poliomielite inativada",
    "Pentavalente",
    "Tríplice Viral",
    "Hepatite B (0 a 30 dias)",
]

chunks_filtrados = []

for chunk in pd.read_csv(caminho, usecols=usecols, chunksize=500_000):
    filtro = (
        chunk["imunizante"].isin(vacinas_mvp)
        & (chunk["ultima_dose"] == True)
        & (chunk["reforco"] == 0)
        & (chunk["ano"] >= 2014)
        & (chunk["ano"] <= 2024)
    )
    chunks_filtrados.append(chunk[filtro])

df = pd.concat(chunks_filtrados, ignore_index=True)

print(f"Total de linhas após filtro: {len(df)}")

grupos = df.groupby(["ibge6", "ano", "imunizante"]).size()
duplicados = grupos[grupos > 1]

print(f"Combinações (ibge6, ano, imunizante) com mais de 1 linha: {len(duplicados)}")

if len(duplicados) > 0:
    ibge6_ex, ano_ex, imunizante_ex = duplicados.index[0]
    exemplo = df[
        (df["ibge6"] == ibge6_ex)
        & (df["ano"] == ano_ex)
        & (df["imunizante"] == imunizante_ex)
    ].head(5)
    print()
    print(f"Exemplo de grupo duplicado: ibge6={ibge6_ex}, ano={ano_ex}, imunizante={imunizante_ex}")
    print(exemplo[["ibge6", "ano", "imunizante", "doenca"]])

C:\Users\Joao Vitor\AppData\Local\Temp\ipykernel_29700\3825656031.py:21: DtypeWarning: Columns (5,8) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(caminho, usecols=usecols, chunksize=500_000):


C:\Users\Joao Vitor\AppData\Local\Temp\ipykernel_29700\3825656031.py:21: DtypeWarning: Columns (5,8) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(caminho, usecols=usecols, chunksize=500_000):


Total de linhas após filtro: 221500


Combinações (ibge6, ano, imunizante) com mais de 1 linha: 0


In [2]:
print(df["ano"].value_counts().sort_index())
print()
print(df["imunizante"].value_counts())

ano
2014    22183
2015    22193
2016    22125
2017    22171
2018    22144
2019    22096
2020    22060
2021    22090
2022    22158
2023    22280
Name: count, dtype: int64

imunizante
Pentavalente              55676
Tríplice Viral            55675
Poliomielite inativada    55657
BCG                       54492
Name: count, dtype: int64


In [ ]:
# Investigação 1: 2024 existe no arquivo bruto (sem filtro nenhum)?
anos_brutos = set()

for chunk in pd.read_csv(caminho, usecols=["ano"], chunksize=500_000):
    anos_brutos.update(chunk["ano"].unique())

print("Valores únicos de 'ano' no arquivo bruto (sem filtro):")
print(sorted(anos_brutos))

# Investigação 2: Hepatite B (0 a 30 dias) — como numero_dose/reforco/ultima_dose se comportam
usecols_hep_b = ["imunizante", "numero_dose", "reforco", "ultima_dose"]

chunks_hep_b = []

for chunk in pd.read_csv(caminho, usecols=usecols_hep_b, chunksize=500_000):
    filtro = chunk["imunizante"] == "Hepatite B (0 a 30 dias)"
    chunks_hep_b.append(chunk[filtro])

df_hep_b = pd.concat(chunks_hep_b, ignore_index=True)

print(f"\nTotal de linhas de 'Hepatite B (0 a 30 dias)' (sem outros filtros): {len(df_hep_b)}")
print("\nnumero_dose:")
print(df_hep_b["numero_dose"].value_counts(dropna=False))
print("\nreforco:")
print(df_hep_b["reforco"].value_counts(dropna=False))
print("\nultima_dose:")
print(df_hep_b["ultima_dose"].value_counts(dropna=False))


In [ ]:
# Filtro final — regra decidida:
# - dose única (BCG, Hepatite B (0 a 30 dias)): sem filtro de ultima_dose/reforco
# - multidose (Pentavalente, Poliomielite inativada, Tríplice Viral): ultima_dose == True e reforco == 0
# Período: 2014 a 2023 (2024 não existe pra essas 5 vacinas no dado bruto)

vacinas_dose_unica = ["BCG", "Hepatite B (0 a 30 dias)"]
vacinas_multidose = ["Pentavalente", "Poliomielite inativada", "Tríplice Viral"]

chunks_final = []

for chunk in pd.read_csv(caminho, usecols=usecols, chunksize=500_000, low_memory=False):
    filtro_periodo = (chunk["ano"] >= 2014) & (chunk["ano"] <= 2023)

    filtro_dose_unica = chunk["imunizante"].isin(vacinas_dose_unica) & filtro_periodo

    filtro_multidose = (
        chunk["imunizante"].isin(vacinas_multidose)
        & (chunk["ultima_dose"] == True)
        & (chunk["reforco"] == 0)
        & filtro_periodo
    )

    chunks_final.append(chunk[filtro_dose_unica | filtro_multidose])

df_processado = pd.concat(chunks_final, ignore_index=True)

print(f"Total de linhas no df_processado: {len(df_processado)}")

caminho_saida = "../data/processed/cobertura_mvp_2014_2023.csv"
df_processado.to_csv(caminho_saida, index=False)
print(f"Salvo em: {caminho_saida}")

In [ ]:
# Validações do CSV processado

print("shape:", df_processado.shape)
print()

print("value_counts de imunizante:")
print(df_processado["imunizante"].value_counts())
print()

print("ano min/max:", df_processado["ano"].min(), "-", df_processado["ano"].max())
print()

grupos = df_processado.groupby(["ibge6", "ano", "imunizante"]).size()
duplicados = grupos[grupos > 1]
print(f"Combinações (ibge6, ano, imunizante) com mais de 1 linha: {len(duplicados)}")
print()

print("Amostra de 5 linhas por vacina:")
print(df_processado.groupby("imunizante").head(5))

In [ ]:
# Dia 3 - Parte 2, Passo 1: carregar o CSV processado com dtypes explícitos
# ibge6 como string (6 dígitos originais, preservados — conversão pro código de 7
# dígitos do IBGE fica pro Dia 6, ao cruzar com o mapa)

df_limpo = pd.read_csv(
    "../data/processed/cobertura_mvp_2014_2023.csv",
    dtype={"ibge6": str, "uf": str, "imunizante": str, "doenca": str},
)

print("dtypes:")
print(df_limpo.dtypes)
print()
df_limpo.info()

In [ ]:
# Passo 2: municípios únicos por ano, por vacina — pra ver se algum ano tem
# queda visível de municípios reportando pra alguma vacina

municipios_por_ano_vacina = (
    df_limpo.groupby(["imunizante", "ano"])["ibge6"]
    .nunique()
    .unstack("ano")
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print(municipios_por_ano_vacina)

In [ ]:
# Passo 3: reler o dicionário de dados por completo (encoding utf-8) e comparar
# com o que aparece de fato no df_limpo

dicionario = pd.read_csv("../data/raw/dicionario-dados-por-municipio.csv", encoding="utf-8")
pd.set_option("display.max_colwidth", 200)
print(dicionario.to_string())
print()

print("`meta` por vacina (valores únicos):")
print(df_limpo.groupby("imunizante")["meta"].unique())
print()

print("`abandono` — % de nulos por vacina:")
print(df_limpo.groupby("imunizante")["abandono"].apply(lambda s: s.isna().mean()))

In [ ]:
# Dia 3 - Parte 3, Passo 1: carregar o CSV processado

df_serie = pd.read_csv(
    "../data/processed/cobertura_mvp_2014_2023.csv",
    dtype={"ibge6": str, "uf": str, "imunizante": str},
)

print(f"Linhas carregadas: {len(df_serie)}")

In [ ]:
# Passo 2: tabela de referência ibge6 -> uf, confirmando que cada município
# só tem um uf no df (se não, isso é problema de qualidade de dado)

ufs_por_municipio = df_serie.groupby("ibge6")["uf"].nunique()
municipios_com_uf_multiplo = ufs_por_municipio[ufs_por_municipio > 1]

print(f"Municípios com mais de um uf diferente: {len(municipios_com_uf_multiplo)}")

if len(municipios_com_uf_multiplo) > 0:
    print(municipios_com_uf_multiplo)
    exemplo_ibge6 = municipios_com_uf_multiplo.index[0]
    print(df_serie[df_serie["ibge6"] == exemplo_ibge6][["ibge6", "uf"]].drop_duplicates())
else:
    ibge6_para_uf = df_serie[["ibge6", "uf"]].drop_duplicates().set_index("ibge6")["uf"]
    print(f"OK — {len(ibge6_para_uf)} municípios únicos, cada um com exatamente um uf.")

In [ ]:
# Passos 3 e 4: grid completo (ibge6 x ano) por vacina, merge com dados reais
# (buracos ficam NaN explícito — decisão tomada), concatenar as 5 vacinas e salvar

vacinas_mvp = [
    "BCG",
    "Hepatite B (0 a 30 dias)",
    "Pentavalente",
    "Poliomielite inativada",
    "Tríplice Viral",
]

todos_ibge6 = sorted(df_serie["ibge6"].unique())
todos_anos = list(range(2014, 2024))

colunas_numericas = [
    "qt_dose", "populacao", "cobertura", "meta", "meta_atingida",
    "abandono", "homog_vacinas", "ultima_dose", "reforco",
]

tabelas_por_vacina = []

for vacina in vacinas_mvp:
    grid = pd.MultiIndex.from_product(
        [todos_ibge6, todos_anos], names=["ibge6", "ano"]
    ).to_frame(index=False)

    dados_vacina = df_serie[df_serie["imunizante"] == vacina][
        ["ibge6", "ano"] + colunas_numericas
    ]

    grid_vacina = grid.merge(dados_vacina, on=["ibge6", "ano"], how="left")
    grid_vacina["imunizante"] = vacina
    grid_vacina["uf"] = grid_vacina["ibge6"].map(ibge6_para_uf)

    tabelas_por_vacina.append(grid_vacina)

df_serie_temporal = pd.concat(tabelas_por_vacina, ignore_index=True)

colunas_finais = ["ibge6", "uf", "ano", "imunizante"] + colunas_numericas
df_serie_temporal = df_serie_temporal[colunas_finais]

print(f"Shape final: {df_serie_temporal.shape}")

caminho_serie = "../data/processed/serie_temporal_mvp.csv"
df_serie_temporal.to_csv(caminho_serie, index=False)
print(f"Salvo em: {caminho_serie}")

In [ ]:
# Passo 5: validações

n_municipios = len(todos_ibge6)
n_vacinas = len(vacinas_mvp)
n_anos = len(todos_anos)
shape_esperado = (n_municipios * n_vacinas * n_anos, len(colunas_finais))

print(f"shape obtido:   {df_serie_temporal.shape}")
print(f"shape esperado: {shape_esperado} ({n_municipios} municípios x {n_vacinas} vacinas x {n_anos} anos)")
print()

nan_cobertura_por_vacina_ano = (
    df_serie_temporal[df_serie_temporal["cobertura"].isna()]
    .groupby(["imunizante", "ano"])
    .size()
    .unstack("ano")
    .fillna(0)
    .astype(int)
)
print("Linhas com cobertura NaN, por vacina e ano:")
print(nan_cobertura_por_vacina_ano)
print()

municipios_que_reportaram = (
    df_serie.groupby(["imunizante", "ano"])["ibge6"].nunique().unstack("ano")
)
buracos_esperados = n_municipios - municipios_que_reportaram
print("Buracos esperados (total de municípios - quem reportou), pra comparar com a tabela acima:")
print(buracos_esperados)
print()

diferenca = (nan_cobertura_por_vacina_ano - buracos_esperados).abs().sum().sum()
print(f"Diferença total entre NaN observado e buraco esperado: {diferenca}")
print()

print(f"uf é NaN em algum lugar do resultado final? {df_serie_temporal['uf'].isna().any()}")

In [ ]:
# Dia 3 - Parte 3 (confirmacao): serie_temporal_mvp.csv ja existia (gerado nas
# celulas anteriores). Antes de assumir que esta correto, recarrega do disco do
# zero, com os dtypes explicitos, e roda as 4 checagens pedidas.

df_serie_temporal = pd.read_csv(
    "../data/processed/serie_temporal_mvp.csv",
    dtype={"ibge6": str, "uf": str, "imunizante": str},
)

vacinas_encontradas = df_serie_temporal["imunizante"].unique()
anos_encontrados = sorted(df_serie_temporal["ano"].unique())
uf_nulos = df_serie_temporal["uf"].isna().sum()
cobertura_nulos = df_serie_temporal["cobertura"].isna().sum()

check_a = len(vacinas_encontradas) == 5
check_b = anos_encontrados == list(range(2014, 2024))
check_c = uf_nulos == 0
check_d = cobertura_nulos > 0

print(f"(a) {len(vacinas_encontradas)} vacinas distintas -> {check_a}")
print(f"    {list(vacinas_encontradas)}")
print(f"(b) anos: {anos_encontrados} -> {check_b}")
print(f"(c) uf nulos: {uf_nulos} -> {check_c}")
print(f"(d) linhas com cobertura nula (buracos esperados): {cobertura_nulos} de {len(df_serie_temporal)} -> {check_d}")
print()
print(f"Arquivo existente passou em todas as checagens, nao precisou reconstruir: {check_a and check_b and check_c and check_d}")

In [ ]:
# Dia 3 - Parte 4, Passo 1: evolucao temporal - media nacional de cobertura por
# ano (2014-2023), uma linha por vacina, ignorando NaN (buracos) na media.

import matplotlib.pyplot as plt

vacinas_mvp = [
    "BCG",
    "Hepatite B (0 a 30 dias)",
    "Pentavalente",
    "Poliomielite inativada",
    "Tríplice Viral",
]
cores_vacina = {
    "BCG": "#2a78d6",
    "Hepatite B (0 a 30 dias)": "#eb6834",
    "Pentavalente": "#1baf7a",
    "Poliomielite inativada": "#eda100",
    "Tríplice Viral": "#e87ba4",
}

media_ano = df_serie_temporal.groupby(["imunizante", "ano"])["cobertura"].mean().unstack("ano")

fig, ax = plt.subplots(figsize=(10, 6), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

for vacina in vacinas_mvp:
    serie = media_ano.loc[vacina]
    ax.plot(
        serie.index, serie.values,
        label=vacina, color=cores_vacina[vacina],
        linewidth=2.2, solid_capstyle="round", marker="o", markersize=4,
    )

ax.set_title("Cobertura vacinal média nacional por ano (2014–2023)", fontsize=13, pad=14)
ax.set_xlabel("Ano", fontsize=10)
ax.set_ylabel("Cobertura média (%)", fontsize=10)
ax.set_xticks(range(2014, 2024))
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_color("#c3c2b7")
ax.legend(frameon=False, fontsize=9, loc="upper right")

fig.tight_layout()
fig.savefig("evolucao_temporal_cobertura.png", dpi=150)
plt.show()

In [ ]:
# Passo 2: comparacao entre estados - media de cobertura por uf no ano mais
# recente (2023), um painel por vacina, estados ordenados do maior pro menor.

df_2023 = df_serie_temporal[df_serie_temporal["ano"] == 2023]
media_uf_2023 = df_2023.groupby(["imunizante", "uf"])["cobertura"].mean()

fig, axes = plt.subplots(2, 3, figsize=(16, 9), facecolor="#fcfcfb")
axes_flat = axes.flatten()

for i, vacina in enumerate(vacinas_mvp):
    ax = axes_flat[i]
    ax.set_facecolor("#fcfcfb")
    ranking = media_uf_2023[vacina].sort_values(ascending=False)
    ax.bar(ranking.index, ranking.values, color=cores_vacina[vacina], width=0.7)
    ax.set_title(vacina, fontsize=11)
    ax.set_ylabel("Cobertura média 2023 (%)", fontsize=8)
    ax.tick_params(axis="x", labelrotation=90, labelsize=6)
    ax.tick_params(axis="y", labelsize=7)
    ax.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color("#c3c2b7")

axes_flat[5].set_visible(False)

fig.suptitle(
    "Cobertura média por estado em 2023, por vacina (ordenado do maior para o menor)",
    fontsize=13, y=0.995,
)
fig.tight_layout()
fig.savefig("comparacao_estados_2023.png", dpi=150)
plt.show()

In [ ]:
# Passo 3: achados - maior/menor cobertura media nacional, tendencia ao longo
# dos 10 anos, e estados que mais aparecem no topo/fundo do ranking em 2023.

media_geral_periodo = media_ano.mean(axis=1).sort_values(ascending=False)
print("Media nacional do periodo 2014-2023, por vacina:")
print(media_geral_periodo.round(1))
print()
print(f"Maior cobertura media no periodo: {media_geral_periodo.index[0]} ({media_geral_periodo.iloc[0]:.1f}%)")
print(f"Menor cobertura media no periodo: {media_geral_periodo.index[-1]} ({media_geral_periodo.iloc[-1]:.1f}%)")
print()

inicio = media_ano[[2014, 2015, 2016]].mean(axis=1)
fim = media_ano[[2021, 2022, 2023]].mean(axis=1)
variacao = (fim - inicio).sort_values()
print("Variacao: media 2021-2023 menos media 2014-2016, por vacina (pontos percentuais):")
print(variacao.round(1))
print()

contagem_topo5 = {}
contagem_fundo5 = {}
for vacina in vacinas_mvp:
    ranking = media_uf_2023[vacina].sort_values(ascending=False)
    for uf in ranking.head(5).index:
        contagem_topo5[uf] = contagem_topo5.get(uf, 0) + 1
    for uf in ranking.tail(5).index:
        contagem_fundo5[uf] = contagem_fundo5.get(uf, 0) + 1

print("Estados mais frequentes no top-5 de 2023 (quantas das 5 vacinas):")
print(sorted(contagem_topo5.items(), key=lambda x: -x[1])[:5])
print()
print("Estados mais frequentes no bottom-5 de 2023 (quantas das 5 vacinas):")
print(sorted(contagem_fundo5.items(), key=lambda x: -x[1])[:5])

In [ ]:
# Ajuste pre-Dia 4: coluna cobertura_capada. `cobertura` pode passar de 100%
# porque a populacao-alvo e uma estimativa (nao uma contagem exata) — isso e
# esperado no dado brasileiro, nao e erro. Mantemos `cobertura` bruta intacta
# pra auditoria e criamos `cobertura_capada` (clip 0-100) pra graficos/modelos.

df_serie_temporal = pd.read_csv(
    "../data/processed/serie_temporal_mvp.csv",
    dtype={"ibge6": str, "uf": str, "imunizante": str},
)

acima_de_100 = df_serie_temporal["cobertura"] > 100

print(f"Total de linhas com cobertura > 100: {acima_de_100.sum()} de {len(df_serie_temporal)}")
print()
print("Por vacina:")
print(df_serie_temporal[acima_de_100].groupby("imunizante").size())

df_serie_temporal["cobertura_capada"] = df_serie_temporal["cobertura"].clip(lower=0, upper=100)

caminho_serie = "../data/processed/serie_temporal_mvp.csv"
df_serie_temporal.to_csv(caminho_serie, index=False)
print()
print(f"Salvo em: {caminho_serie}, agora com {list(df_serie_temporal.columns)}")